<a href="https://colab.research.google.com/github/abrown12005/CMP_SC-4540-HW/blob/main/HW7/EEGClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

RunningInCOLAB = 'google.colab' in str(get_ipython())
if RunningInCOLAB:
    # clone github repo with the data
    !git clone https://github.com/abrown12005/CMP_SC-4540-HW.git
    %cd CMP_SC-4540-HW/HW7

data = pd.read_csv('EEG_data.csv')

# This data is EEG brainwave signals from students watching confusion videos
# The inputs are the signals and the output is whether the student is confused
# This is a Classification problem for Confused or not-Confused
# First use Logisitic Regression
X = np.array(data.iloc[1:-1, 2:-2])
y = np.array(data.iloc[1:-1, -1])

model = LogisticRegression(max_iter=1000)
model.fit(X,y)
pedictions = model.predict(X)
accuracy = accuracy_score(y, pedictions)
print("Logisitc Regression Accuracy:", accuracy)

# Now use Neural Network
n_X = torch.tensor(X, dtype=torch.float32)
n_y = torch.tensor(y, dtype=torch.float32).reshape(-1,1)

model = nn.Sequential(
    nn.Linear(n_X.shape[1],1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1000):
    pred = model(n_X)
    loss = criterion(pred, n_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Accuracy
with torch.no_grad():
    predicted = (model(n_X) > 0.5).float()
    accuracy = (predicted == n_y).float().mean()

print("Neural Network Accuracy:", accuracy.item())

# Run the Neural Network again after performing feature engineering

# Only use a subset of the features (only use the alpha,beta,gamma waves)
# Only use the first half of Subject ID's
X_sub = X[1:6457,7:-1]
y_sub = y[1:6457]

# Remove outliers using the IQR boundary
Q1 = np.percentile(X_sub, 25, axis=0)
Q3 = np.percentile(X_sub, 75, axis=0)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

mask = np.all((X_sub >= lower) & (X_sub <= upper), axis=1)

X_clean = X_sub[mask]
y_clean = y_sub[mask]

# Normalize the data
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_clean)
f_X = torch.tensor(X_scaled, dtype=torch.float32)
f_y = torch.tensor(y_clean, dtype=torch.float32).reshape(-1,1)

f_model = nn.Sequential(
    nn.Linear(f_X.shape[1],1),
    nn.Sigmoid()
)

for epoch in range(1000):
    pred = f_model(f_X)
    loss = criterion(pred, f_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Accuracy
with torch.no_grad():
    predicted = (f_model(f_X) > 0.5).float()
    accuracy = (predicted == f_y).float().mean()

print("New-Neural Network Accuracy:", accuracy.item())

Cloning into 'CMP_SC-4540-HW'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 274 (delta 49), reused 0 (delta 0), pack-reused 151 (from 1)
Receiving objects: 100% (274/274), 23.36 MiB | 10.27 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7/CMP_SC-4540-HW/HW7
Logisitc Regression Accuracy: 0.6003591224919979
Neural Network Accuracy: 0.5126863718032837
New-Neural Network Accuracy: 0.5235127210617065
